# Support Vector Machine (SVM) Trading Strategy

This notebook implements and tests a Support Vector Machine (SVM) classification model for stock trading. The methodology is inspired by the classic paper:

**"Financial time series forecasting using support vector machines"** by *Kyoung-jae Kim* (Neurocomputing, 2003).

### Strategy Overview
- **Objective**: Predict the daily direction of change in the stock price (binary classification: 1 for UP, 0 for DOWN).
- **Features**: A set of key technical indicators derived from daily price data (SMA, Volatility, daily returns, RSI, MACD).
- **Labels**: 1 if the next day's close price is higher than today's close, 0 otherwise.
- **Tuning**: Grid search over SVM hyper-parameters (regularization $C$, kernel type, and kernel parameter $\gamma$).

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

# Setup paths to import portfolio package
base_dir = os.path.abspath("../../..")
if base_dir not in sys.path:
    sys.path.append(base_dir)

portfolio_repo_path = os.path.join(base_dir, "portfolio-management")
if portfolio_repo_path not in sys.path:
    sys.path.append(portfolio_repo_path)

from portfolio.models.market import Market
from portfolio.models.features import Features
from portfolio.models.metrics import Metrics

## 1. Load Data and Extract Features
We load historical daily data for Apple (AAPL) from our SQLite market database and compute the technical indicators.

In [ ]:
def get_data(ticker="AAPL"):
    # Retrieve data using the portfolio-management Market module
    df = Market.get_historical_data(tickers=[ticker])
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")
    
    close = df["price_close"]
    
    # Calculate technical indicators
    sma_20 = Features.get_moving_average(close, 20)
    sma_50 = Features.get_moving_average(close, 50)
    
    log_returns = Metrics.get_daily_log_returns(close)
    rolling_vol = log_returns.rolling(window=20).std()
    
    daily_returns = Metrics.get_daily_returns(close)
    
    # RSI 14
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(com=13, adjust=False).mean()
    avg_loss = loss.ewm(com=13, adjust=False).mean()
    rs = avg_gain / (avg_loss + 1e-9)
    rsi_14 = 100 - (100 / (1 + rs))
    
    # MACD
    ema_12 = close.ewm(span=12, adjust=False).mean()
    ema_26 = close.ewm(span=26, adjust=False).mean()
    macd = ema_12 - ema_26
    macd_signal = macd.ewm(span=9, adjust=False).mean()
    macd_hist = macd - macd_signal
    
    # Target: 1 if next day close > today's close, else 0
    target = (close.shift(-1) > close).astype(int)
    
    data_df = pd.DataFrame({
        "Close": close,
        "SMA_20": sma_20,
        "SMA_50": sma_50,
        "Volatility_20": rolling_vol,
        "returns": daily_returns,
        "RSI_14": rsi_14,
        "MACD": macd,
        "MACD_Signal": macd_signal,
        "MACD_Hist": macd_hist,
        "Target": target
    })
    
    data_df.dropna(inplace=True)
    return data_df

df = get_data("AAPL")
print(f"Data points available: {len(df)}")
df.head()

## 2. Train-Test Split and Standardization

In [ ]:
feature_cols = ["SMA_20", "SMA_50", "Volatility_20", "returns", "RSI_14", "MACD", "MACD_Signal", "MACD_Hist"]

X = df[feature_cols].values
y = df["Target"].values

# Temporal split (80% train, 20% test)
split_idx = int(len(df) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

## 3. Parameter Sweep / Hyperparameter Tuning

In [ ]:
c_list = [0.1, 1.0, 10.0, 100.0]
gamma_list = ["scale", "auto", 0.01, 0.1, 1.0]
kernel_list = ["rbf", "linear"]

results = []
market_returns = df["returns"].values[split_idx:]

for kernel in kernel_list:
    for C in c_list:
        for gamma in gamma_list:
            if kernel == "linear" and gamma != "scale":
                continue
                
            model = SVC(kernel=kernel, C=C, gamma=gamma, random_state=42)
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)
            
            acc = accuracy_score(y_test, y_pred)
            prec = precision_score(y_test, y_pred, zero_division=0)
            rec = recall_score(y_test, y_pred, zero_division=0)
            f1 = f1_score(y_test, y_pred, zero_division=0)
            
            # Shift signals to avoid lookahead bias
            positions = np.zeros_like(y_pred)
            positions[1:] = y_pred[:-1]
            
            strategy_returns = positions * market_returns
            
            cumulative_market = np.prod(1 + market_returns) - 1
            cumulative_strategy = np.prod(1 + strategy_returns) - 1
            
            mean_ret = np.mean(strategy_returns)
            std_ret = np.std(strategy_returns)
            sharpe = (mean_ret / (std_ret + 1e-9)) * np.sqrt(252)
            
            cum_prod = np.cumprod(1 + strategy_returns)
            running_max = np.maximum.accumulate(cum_prod)
            drawdowns = (cum_prod - running_max) / (running_max + 1e-9)
            max_dd = np.min(drawdowns)
            
            results.append({
                "Kernel": kernel,
                "C": C,
                "Gamma": gamma,
                "Accuracy": acc,
                "Precision": prec,
                "Recall": rec,
                "F1-Score": f1,
                "Market Return": cumulative_market,
                "Strategy Return": cumulative_strategy,
                "Sharpe Ratio": sharpe,
                "Max Drawdown": max_dd
            })
            
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="Strategy Return", ascending=False)
results_df.head(10)

## 4. Evaluate and Plot Best Model vs. Buy and Hold
We train the model with the best parameters (Kernel: `rbf`, $C$: `1.0`, $\gamma$: `'auto'`) and plot the backtest cumulative return.

In [ ]:
best_model = SVC(kernel="rbf", C=1.0, gamma="auto", random_state=42)
best_model.fit(X_train_scaled, y_train)
y_pred = best_model.predict(X_test_scaled)

print("Best Model Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Down (0)", "Up (1)"]))

positions = np.zeros_like(y_pred)
positions[1:] = y_pred[:-1]

strategy_returns = positions * market_returns

cum_market = np.cumprod(1 + market_returns) - 1
cum_strategy = np.cumprod(1 + strategy_returns) - 1

test_dates = df.index[split_idx:]

plt.figure(figsize=(12, 6))
plt.plot(test_dates, cum_market * 100, label="Market (Buy & Hold)", color="#e74c3c", lw=2)
plt.plot(test_dates, cum_strategy * 100, label="SVM Strategy (RBF, C=1.0, gamma=auto)", color="#2ecc71", lw=2)
plt.title("AAPL Backtest: SVM Strategy vs. Market", fontsize=14, fontweight="bold")
plt.xlabel("Date", fontsize=12)
plt.ylabel("Cumulative Return (%)", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(fontsize=11)
plt.show()